# Transformer 端到端模型 —— 加载 JSON 权重 + 推理打分

本 notebook 只负责**推理**：加载 `Transformer_modelsave_train.py` 训练产出的 `transformer_model.json`，在平台注入的测试区间上打分，**不训练**。

- 模型结构、切窗、数值变换、标准化全部从 `Transformer_modelsave_train.py` 导入，与训练完全同一份代码；标准化统计（mean/std）与变换方式（norm）随权重一起存于 JSON。
- 推理按月分片切窗、预测，避免整段测试区间常驻内存。
- 输出 `date / instrument / factor` 三列，只保留 `cpt_jyc_2026_instruments` 中的 8 个日内截面。

提交：`Transformer_modelsave_train.py` + `transformer_model.json` + 本 notebook 一并打包上传。

In [ ]:
import os

import numpy as np
import pandas as pd
import dai
import torch
import structlog

from Transformer_modelsave_train import (
    MODEL_PATH, StockTransformer, load_model, pool, pick_device, predict_range,
)

logger = structlog.get_logger()


def main(datasources, start_date, end_date):
    """加载已训练好的模型，在测试区间 start_date~end_date 上推理打分，输出 ['date', 'instrument', 'factor']。"""
    table = datasources["bar1m"]
    device = pick_device()
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"未找到模型文件 {MODEL_PATH}，请随 notebook 一起上传 transformer_model.json")

    ckpt = load_model(MODEL_PATH, map_location=device)
    stats = (np.asarray(ckpt["mean"], np.float32), np.asarray(ckpt["std"], np.float32))
    model = StockTransformer(**ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    logger.info("已加载模型", path=MODEL_PATH, device=str(device), norm=ckpt["norm"])

    instruments = pool(start_date, end_date)
    pred = predict_range(model, table, start_date, end_date, instruments, stats, device, norm=ckpt["norm"])

    # 对齐比赛股票池与 8 个日内截面，规范输出
    stk = dai.query("SELECT date, instrument FROM cpt_jyc_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    stk["date"] = pd.to_datetime(stk["date"])
    pred["date"] = pd.to_datetime(pred["date"])
    result = (pd.merge(stk, pred, on=["date", "instrument"], how="inner")
                .replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])
                .drop_duplicates(["date", "instrument"])[["date", "instrument", "factor"]]
                .sort_values(["date", "instrument"])
                .reset_index(drop=True))
    logger.info("因子构建完成", rows=len(result), dates=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result
